# 练习实验：使用 Neural Networks 识别手写数字——Multiclass Classification

本练习将使用 Neural Network 识别手写数字 0–9。

## 目录

- [1 - 软件包](#1)
- [2 - ReLU Activation](#2)
- [3 - Softmax Function](#3)
  - [练习 1](#ex01)
- [4 - Neural Networks](#4)
  - [4.1 问题描述](#4.1)
  - [4.2 数据集](#4.2)
  - [4.3 模型表示](#4.3)
  - [4.4 使用 TensorFlow 实现模型](#4.4)
  - [4.5 Softmax 的放置位置](#4.5)
    - [练习 2](#ex02)


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除本 Notebook 中不参与评分的单元，也不要添加新单元。
>
> **通过本作业后**，如果想尝试修改不参与评分的代码，可以按照 Notebook 底部的说明操作。


<a name="1"></a>
## 1 - 导入依赖库

首先运行下面的代码单元，导入本作业所需的库。

- [NumPy](https://numpy.org/)：Python 的基础科学计算库。
- [Matplotlib](http://matplotlib.org)：常用的 Python 绘图库。
- [TensorFlow](https://www.tensorflow.org/)：常用的 Machine Learning 平台。


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

from public_tests import * 

from autils import *
from lab_utils_softmax import plt_softmax
np.set_printoptions(precision=2)

<a name="2"></a>
## 2 - ReLU Activation

本周介绍了新的 activation function：Rectified Linear Unit（ReLU）。

$$
a=\max(0,z)
$$


In [ ]:
plt_act_trio()

<img align="right" src="./images/C2_W2_ReLu.png" style="width:380px; padding:10px 20px;">

右侧的课程示例展示了 ReLU 的应用。派生的 `awareness` 特征不是二值变量，而是具有连续的取值范围。Sigmoid 适合开/关或二分类场景；ReLU 则提供连续的线性关系，并具有输出为零的“关闭”区间。

这个“关闭”区间使 ReLU 成为 Non-linear Activation，使多个单元能共同构成目标函数，并避免在不需要的区间相互干扰。配套的选做实验将进一步探讨这一点。


<a name="3"></a>
## 3 - Softmax Function

用于 Multiclass Classification 的 Neural Network 会生成 $N$ 个输出，并从中选出预测结果。Output Layer 先通过线性函数生成向量 $\mathbf{z}$，再将它传入 Softmax。Softmax 将 $\mathbf{z}$ 转换为 Probability Distribution：每个输出都在 0 到 1 之间，且总和为 1，因此可解释为概率。较大的输入对应较大的输出概率。

<div style="text-align:center;"><img src="./images/C2_W2_NNSoftmax.PNG" width="600" alt="Neural Network 的 Softmax 输出"></div>


Softmax Function 可写为：

$$a_j=\frac{e^{z_j}}{\sum_{k=0}^{N-1}e^{z_k}}.\tag{1}$$

其中 $z_j=\mathbf{w}_j\cdot\mathbf{x}+b_j$，$N$ 是 Output Layer 中的类别数。


<a name="ex01"></a>
### 练习 1

下面使用 NumPy 实现 Softmax。


In [ ]:
# UNQ_C1
# GRADED CELL: my_softmax

def my_softmax(z):  
    """ Softmax converts a vector of values to a probability distribution.
    Args:
      z (ndarray (N,))  : input data, N features
    Returns:
      a (ndarray (N,))  : softmax of z
    """    
    ### START CODE HERE ### 
    
    
    
    
        
    
        
    
    
    ### END CODE HERE ### 
    return a

In [ ]:
z = np.array([1., 2., 3., 4.])
a = my_softmax(z)
atf = tf.nn.softmax(z)
print(f"my_softmax(z):         {a}")
print(f"tensorflow softmax(z): {atf}")

# BEGIN UNIT TEST  
test_my_softmax(my_softmax)
# END UNIT TEST  

<details>
<summary><strong>点击查看提示</strong></summary>

一种实现方式是先用一个 `for` 循环计算分母，再用另一个循环计算各个输出。

```python
def my_softmax(z):  
    N = len(z)
    a =                     # initialize a to zeros 
    ez_sum =                # initialize sum to zero
    for k in range(N):      # loop over number of outputs             
        ez_sum +=           # sum exp(z[k]) to build the shared denominator      
    for j in range(N):      # loop over number of outputs again                
        a[j] =              # divide each the exp of each output by the denominator   
    return(a)
```

<details>
<summary><strong>点击查看代码</strong></summary>

```python
def my_softmax(z):
    N = len(z)
    a = np.zeros(N)
    ez_sum = 0
    for k in range(N):
        ez_sum += np.exp(z[k])
    for j in range(N):
        a[j] = np.exp(z[j])/ez_sum
    return(a)
```

也可以使用 Vectorization：

```python
def my_softmax(z):
    ez = np.exp(z)
    a = ez/np.sum(ez)
    return(a)
```

</details>
</details>


在下方调整输入 `z` 的值。特别留意分子中的指数运算如何放大较小的输入差异，以及输出值之和始终为 1。


In [ ]:
plt.close("all")
plt_softmax(my_softmax)

<a name="4"></a>
## 4 - Neural Networks

上周的作业使用 Neural Network 完成 Binary Classification；本周将借助 Softmax Activation 将它扩展到 Multiclass Classification。

<a name="4.1"></a>
### 4.1 问题描述

本练习将识别 0–9 这十种手写数字。这是一个从多个候选类别中选择一个类别的 Multiclass Classification 任务。自动手写数字识别有广泛应用，例如识别信封上的邮政编码或银行支票上的金额。

<a name="4.2"></a>
### 4.2 数据集

首先加载数据集。下面的 `load_data()` 函数将数据加载到 `X` 和 `y`。

- 数据集包含 5,000 个手写数字训练样本¹，每个样本是一幅 $20\times20$ 像素的灰度图像。
- 每个像素用一个浮点数表示该位置的灰度强度。将 $20\times20$ 的像素网格展开，得到一个 400 维向量。
- 每个样本对应数据矩阵 `X` 的一行，因此 `X` 的形状为 `(5000, 400)`：

$$X=\begin{bmatrix}
\text{---}\;(x^{(1)})\;\text{---}\\
\text{---}\;(x^{(2)})\;\text{---}\\
\vdots\\
\text{---}\;(x^{(m)})\;\text{---}
\end{bmatrix}.$$

训练集的另一部分是形状为 `(5000, 1)` 的标签向量 `y`。图像中的数字为 `0` 时，标签为 `0`；数字为 `4` 时，标签为 `4`，其余依此类推。

¹ 本数据集是 [MNIST 手写数字数据集](http://yann.lecun.com/exdb/mnist/) 的一个子集。


In [ ]:
# load dataset
X, y = load_data()

#### 4.2.1 查看变量

先打印变量的内容，熟悉数据集。下面的代码输出 `X` 和 `y` 的第一个元素。


In [ ]:
print ('The first element of X is: ', X[0])

In [ ]:
print ('The first element of y is: ', y[0,0])
print ('The last element of y is: ', y[-1,0])

#### 4.2.2 检查变量维度

查看数据的维度也有助于理解数据集。打印 `X` 和 `y` 的形状，确认训练样本的数量。


In [ ]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

#### 4.2.3 数据可视化

下面展示训练集的一部分：代码从 `X` 中随机选择 64 行，将每行还原为 $20\times20$ 像素的灰度图像并集中显示。每幅图像上方显示对应标签。


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[left, bottom, right, top]

#fig.tight_layout(pad=0.5)
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Display the label above the image
    ax.set_title(y[random_index,0])
    ax.set_axis_off()
    fig.suptitle("Label, image", fontsize=14)

<a name="4.3"></a>
### 4.3 模型表示

本作业使用的 Neural Network 如下图所示：前两个 Dense Layer 使用 ReLU Activation，最后的 Output Layer 使用 Linear Activation。输入是数字图像的像素值；每幅图像为 $20\times20$ 像素，因此共有 400 个输入。

<img src="images/C2_W2_Assigment_NN.png" width="600" height="450" alt="手写数字分类网络结构">


网络第 1 层有 25 个单元，第 2 层有 15 个单元，第 3 层有 10 个输出单元，分别对应十种数字。

如果当前层有 $s_{in}$ 个单元，下一层有 $s_{out}$ 个单元，则 Weight Matrix $W$ 的形状为 $s_{in}\times s_{out}$，Bias Vector $b$ 包含 $s_{out}$ 个元素。因此，各层参数的形状为：

| 层 | Weight Matrix | Bias Vector |
| --- | --- | --- |
| 第 1 层 | `W1`: `(400, 25)` | `b1`: `(25,)` |
| 第 2 层 | `W2`: `(25, 15)` | `b2`: `(15,)` |
| 第 3 层 | `W3`: `(15, 10)` | `b3`: `(10,)` |

> **注意：** Bias Vector 可以用一维数组 `(n,)` 或二维列向量 `(n, 1)` 表示。TensorFlow 使用一维表示，本实验也遵循这一约定。


<a name="4.4"></a>
### 4.4 使用 TensorFlow 实现模型


TensorFlow 模型按 layer 构建。你指定每层的*输出维度*，框架据此计算下一层的输入维度。第一层的输入维度可以在 `model.fit` 接收数据时推断。

也可以显式指定输入，例如：

```python
tf.keras.Input(shape=(400,))
```

本实验采用显式输入，以便提前查看参数的 shape。


<a name="4.5"></a>
### 4.5 Softmax 的放置位置

课程和 Softmax 选做实验介绍过：训练时将 Softmax 与 Loss Function 合并计算，可以提高 Numerical Stability。这会影响模型的构建与使用。

构建模型时：

- 最后的 Dense Layer 使用 `linear` Activation，相当于不进行非线性激活。
- 在 `model.compile` 中设置 `loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)`，表示输入 Loss Function 的是 Logits。
- 目标标签的形式不变。使用 `SparseCategoricalCrossentropy` 时，标签仍是期望识别出的数字 0–9。

使用模型时，模型直接输出的值不是概率；如需概率，需再应用 Softmax。


<a name="ex02"></a>
### 练习 2

使用 Keras 的 [Sequential model](https://keras.io/guides/sequential_model/) 和 [Dense Layer](https://keras.io/api/layers/core_layers/dense/) 构建上述三层网络：前两层使用 ReLU，最后一层使用 Linear Activation。


In [ ]:
# UNQ_C2
# GRADED CELL: Sequential model
tf.random.set_seed(1234) # for consistent results
model = Sequential(
    [               
        ### START CODE HERE ### 
        
        
        
        
        ### END CODE HERE ### 
    ], name = "my_model" 
)

In [ ]:
model.summary()

<details>
<summary><strong>预期输出（点击展开）</strong></summary>

`model.summary()` 会显示模型摘要。如果未显式指定层名，名称会自动生成，因此可能与下方不同。

```
Model: "my_model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 25)                10025     
_________________________________________________________________
L2 (Dense)                   (None, 15)                390       
_________________________________________________________________
L3 (Dense)                   (None, 10)                160       
=================================================================
Total params: 10,575
Trainable params: 10,575
Non-trainable params: 0
_________________________________________________________________
```

</details>


<details>
<summary><strong>点击查看提示</strong></summary>

```python
tf.random.set_seed(1234)
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.Input(shape=(400,)),     # @REPLACE 
        Dense(25, activation='relu', name = "L1"), # @REPLACE 
        Dense(15, activation='relu',  name = "L2"), # @REPLACE  
        Dense(10, activation='linear', name = "L3"),  # @REPLACE 
        ### END CODE HERE ### 
    ], name = "my_model" 
)
```

</details>


In [ ]:
# BEGIN UNIT TEST     
test_model(model, 10, 400)
# END UNIT TEST     

摘要中的参数数量，对应各个 weight 和 bias array 的元素总数，如下所示。


下面进一步检查 weight，确认 TensorFlow 创建的参数 shape 与前面的计算一致。


In [ ]:
[layer1, layer2, layer3] = model.layers

In [ ]:
#### Examine Weights shapes
W1,b1 = layer1.get_weights()
W2,b2 = layer2.get_weights()
W3,b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")

**预期输出**

```
W1 shape = (400, 25), b1 shape = (25,)  
W2 shape = (25, 15), b2 shape = (15,)  
W3 shape = (15, 10), b3 shape = (10,)
```


以下代码定义：

- Loss Function `SparseCategoricalCrossentropy`，并设置 `from_logits=True`，将 Softmax 纳入 Loss 的计算。
- Optimizer。本例使用课程中介绍的 Adam（Adaptive Moment Estimation），这是一种常用的优化算法。


In [ ]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
)

history = model.fit(
    X,y,
    epochs=40
)

#### Epochs 与 Batches

上面的 `fit` 将 `epochs` 设为 40，表示训练过程中遍历整个数据集 40 次。训练进度输出类似于：

```
Epoch 1/40
157/157 [==============================] - 0s 1ms/step - loss: 2.2770
```

第一行 `Epoch 1/40` 表示当前进行到哪个 Epoch。为提高效率，训练集被划分为多个 Batch。此处未指定 Batch Size 时使用 32；5,000 个样本需要约 157 个 Batch。第二行的 `157/157 [====` 表示已完成的 Batch 数量。


#### Loss（Cost）

Course 1 介绍了通过监控 Cost 来跟踪 Gradient Descent 的进展。理想情况下，Cost 会随着迭代次数增加而下降。TensorFlow 将这里的 Cost 称为 `loss`；上面的 `model.fit` 会在每个 Epoch 显示它。

[`.fit`](https://www.tensorflow.org/api_docs/python/tf/keras/Model) 方法返回包含 Loss 等指标的历史记录，上方将其保存在 `history` 中。下面利用这些记录绘制 Loss 曲线。


In [ ]:
plot_loss_tf(history)

#### Prediction

使用 Keras 的 `predict` 进行预测。下面的 `X[1015]` 是一幅数字 2 的图像。


In [ ]:
image_of_two = X[1015]
display_digit(image_of_two)

prediction = model.predict(image_of_two.reshape(1,400))  # prediction

print(f" predicting a Two: \n{prediction}")
print(f" Largest Prediction index: {np.argmax(prediction)}")

最大的输出是 `prediction[2]`，表示预测数字为 2。如果只需选择类别，使用 NumPy 的 [`argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 即可。如果还需要概率，则要应用 Softmax：


In [ ]:
prediction_p = tf.nn.softmax(prediction)

print(f" predicting a Two. Probability vector: \n{prediction_p}")
print(f"Total of predictions: {np.sum(prediction_p):0.3f}")

要返回表示预测类别的整数，应取最大概率所在的索引。可使用 NumPy 的 [`argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 实现。


In [ ]:
yhat = np.argmax(prediction_p)

print(f"np.argmax(prediction_p): {yhat}")

下面随机选择 64 个数字，比较预测结果与 label。运行可能需要一点时间。


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[left, bottom, right, top]
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Predict using the Neural Network
    prediction = model.predict(X[random_index].reshape(1,400))
    prediction_p = tf.nn.softmax(prediction)
    yhat = np.argmax(prediction_p)
    
    # Display the label above the image
    ax.set_title(f"{y[random_index,0]},{yhat}",fontsize=10)
    ax.set_axis_off()
fig.suptitle("Label, yhat", fontsize=14)
plt.show()

下面查看一些预测错误的样本。

> **注意：** 增加训练 Epoch 数可能消除本数据集中的这些错误。


In [ ]:
print( f"{display_errors(model,X,y)} errors out of {len(X)} images")

### 恭喜完成！

你已经成功构建并使用 Neural Network 完成 Multiclass Classification。


<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="../../../assets/images/unlock_cells.gif" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
